# Phase 2: Feature Engineering and Resampling

This notebook builds the Final Feature Matrix from the Gold Dataset (8,268 seniors) using hrp_processed.db. We'll:

1. **Resample** vitals into 15-minute buckets (mean for HR/BP/Temp/Sat, sum for Steps)
2. **Pivot** from long to wide format
3. **Impute** missing values with forward fill (1 hour limit)
4. **Engineer features**: hr_volatility, bp_trend, pulse_pressure
5. **Fuse static data**: 11 clinical domain flags
6. **Create alert context**: recent_event_burden (Severity 1/2 alerts in past 48h)
7. **Label targets**: label_1 / label_2 / label_3 = 1 if within 24h before Severity 1 / 2 / 3 alerts
8. **Save output**: data/processed/multimodal_features.parquet

Processing senior-by-senior to manage memory for the large final matrix.

## Section 1: Import Libraries

In [43]:
import time
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import timedelta
import warnings
import pickle
import json
warnings.filterwarnings('ignore')

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (14, 8)

## Section 2: Load Gold Dataset and Database Connection

In [25]:
db_path = '../data/processed/hrp_processed.db'
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

print(f"Connected to: {db_path}")

Connected to: ../data/processed/hrp_processed.db


In [32]:
gold_dataset_pickle_path = '../data/processed/gold_seniors.pkl'
gold_dataset_csv_path = '../data/processed/gold_seniors.csv'
metadata_path = '../data/processed/gold_dataset_metadata.json'

with open(gold_dataset_pickle_path, 'rb') as f:
    gold_seniors = pickle.load(f)

gold_seniors_list = list(gold_seniors)

In [38]:
len(gold_seniors)

8268

In [30]:
with open(metadata_path, 'r') as f:
    gold_metadata = json.load(f)

print("Gold Dataset Metadata:")
print(f"  - Total Seniors: {gold_metadata['total_seniors']:,}")
print(f"  - Retention Rate: {gold_metadata['retention_rate_percent']:.1f}%")
print(f"  - Severity 3 Alerts Retained: {gold_metadata['severity_3_alerts']:,} ({gold_metadata['severity_3_retention_percent']:.1f}%)")
print("  - Criteria:")
print(f"    * Criterion 1 ({gold_metadata['criterion_1_count']:,}): {gold_metadata['criteria_description']['criterion_1']}")
print(f"    * Criterion 2 ({gold_metadata['criterion_2_count']:,}): {gold_metadata['criteria_description']['criterion_2']}")

Gold Dataset Metadata:
  - Total Seniors: 8,268
  - Retention Rate: 64.6%
  - Severity 3 Alerts Retained: 47 (77.0%)
  - Criteria:
    * Criterion 1 (8,267): Seniors with >30% overall data density
    * Criterion 2 (26): Seniors with >80% local density in 6 hours before Severity 3 alert


## Section 3: Create Database Index

Before processing 8,268 seniors with 70M+ measurements, we need an index on senior_id to avoid full table scans.

In [41]:
index_check = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"Existing indexes: {[idx[0] for idx in index_check]}")

try:
    cursor.execute("CREATE INDEX IF NOT EXISTS idx_measurements_senior_id ON measurements(senior_id)")
    conn.commit()
except Exception as e:
    print(f"Index creation: {e}")

index_verify = cursor.execute("SELECT name FROM sqlite_master WHERE type='index' AND tbl_name='measurements'").fetchall()
print(f"\nCurrent indexes: {[idx[0] for idx in index_verify]}")

Existing indexes: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date', 'idx_measurements_senior_id']

Current indexes: ['idx_measurements_senior', 'idx_measurements_date', 'idx_measurements_senior_date', 'idx_measurements_senior_id']


## Section 4: Resample Vitals into 15-Minute Buckets

In [47]:
def resample_senior_vitals(senior_id, conn):
    """Resample a single senior's measurements into 15-minute buckets."""
    query = """
    SELECT date, type, value, sbp, dbp, pulse_pressure
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date
    """
    df = pd.read_sql_query(query, conn, params=(str(senior_id),))
    if len(df) == 0:
        return None

    df['date'] = pd.to_datetime(df['date'])
    agg_funcs = {'Heartrate': 'mean', 'Temperature': 'mean', 'Saturation': 'mean', 'Steps': 'sum'}
    df.rename(columns={'type': 'measurement_type'}, inplace=True)
    df.set_index('date', inplace=True)

    resampled_dfs = []
    for mtype in df['measurement_type'].unique():
        df_type = df[df['measurement_type'] == mtype].copy()
        if mtype == 'BloodPressure':
            resampled_dfs.append(df_type['sbp'].resample('15min').mean().to_frame(name='sbp'))
            resampled_dfs.append(df_type['dbp'].resample('15min').mean().to_frame(name='dbp'))
            if 'pulse_pressure' in df_type.columns:
                resampled_dfs.append(df_type['pulse_pressure'].resample('15min').mean().to_frame(name='pulse_pressure'))
        else:
            agg_func = agg_funcs.get(mtype, 'mean')
            resampled_dfs.append(df_type['value'].resample('15min').agg(agg_func).to_frame(name=mtype.lower()))

    if len(resampled_dfs) == 0:
        return None

    result = pd.concat(resampled_dfs, axis=1).reset_index()
    result['senior_id'] = senior_id
    return result

## Section 5: Pivot Data from Long to Wide Format

In [48]:
def pivot_to_wide_format(df_resampled):
    """Transform resampled data from long to wide format."""
    if df_resampled is None or len(df_resampled) == 0:
        return None
    
    df_wide = df_resampled.copy()
    df_wide.rename(columns={'date': 'timestamp'}, inplace=True)
    
    df_wide.columns = [col.lower().replace(' ', '_') for col in df_wide.columns]
    
    return df_wide

## Section 6: Apply Forward Fill Imputation

In [49]:
def apply_forward_fill_imputation(df_wide, limit_buckets=4):
    """Apply forward fill imputation with a limit of 4 buckets (1 hour at 15-min intervals)."""
    if df_wide is None or len(df_wide) == 0:
        return df_wide
    
    df_filled = df_wide.copy()
    
    vital_cols = [col for col in df_filled.columns 
                  if col not in ['timestamp', 'senior_id']]
    
    for col in vital_cols:
        df_filled[col] = df_filled[col].fillna(method='ffill', limit=limit_buckets)
    
    return df_filled

## Section 7: Engineer Signal Features (HR Volatility & BP Trend)

In [53]:
def engineer_signal_features(df_filled):
    """
    Add engineered signal features:
    - hr_volatility: 4-hour rolling standard deviation of HR (16 buckets at 15-min)
    - bp_trend: Slope of SBP over last 3 hours (12 buckets at 15-min)
    """
    
    if df_filled is None or len(df_filled) == 0:
        return df_filled
    
    df_features = df_filled.copy()
    
    if 'heartrate' in df_features.columns:
        df_features['hr_volatility'] = df_features['heartrate'].rolling(
            window=16, min_periods=1
        ).std()
    
    def calculate_slope(series):
        if len(series) < 2:
            return np.nan
        x = np.arange(len(series))
        mask = ~np.isnan(series)
        if mask.sum() < 2:
            return np.nan
        slope, _ = np.polyfit(x[mask], series[mask], 1)
        return slope
    
    if 'sbp' in df_features.columns:
        df_features['bp_trend'] = df_features['sbp'].rolling(
            window=12, min_periods=2
        ).apply(calculate_slope, raw=False)
    
    return df_features

## Section 8: Fuse Static Clinical Domain Flags

In [54]:
def load_clinical_domain_flags(senior_id, conn):
    """Load the 11 clinical domain flags from the senior_risk_profiles table."""
    
    query = """
    SELECT * FROM senior_risk_profiles WHERE senior_id = ?
    """
    
    df_flags = pd.read_sql_query(query, conn, params=(str(senior_id),))
    
    if len(df_flags) == 0:
        return None
    
    flag_cols = [col for col in df_flags.columns 
                 if col != 'senior_id' and df_flags[col].dtype in ['int64', 'float64', 'bool']]
    
    return df_flags[['senior_id'] + flag_cols].iloc[0]

def fuse_clinical_flags(df_features, clinical_flags):
    """Join clinical domain flags to each time bucket."""
    
    if df_features is None or clinical_flags is None:
        return df_features
    
    df_fused = df_features.copy()
    
    for col in clinical_flags.index:
        if col != 'senior_id':
            df_fused[col] = clinical_flags[col]
    
    return df_fused

## Section 9: Create Alert Context Feature (Recent Event Burden)

In [55]:
def add_alert_context_feature(df_features, senior_id, conn):
    """Create recent_event_burden features for each time bucket."""
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_context = df_features.copy()
    
    query = """
    SELECT alert_date FROM alerts 
    WHERE senior_id = ? AND severity IN (1, 2, 3)
    ORDER BY alert_date
    """
    alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id),))

    df_features['recent_event_burden'] = 0.0
    
    if len(alerts_df) == 0:
        return df_context
    
    alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])

    alert_series = pd.Series(0, index=df_features['timestamp'])
    alert_times = alerts_df['alert_date'].dt.round('15min')
    alert_counts = alert_times.value_counts()

    common_indices = alert_series.index.intersection(alert_counts.index)
    alert_series.loc[common_indices] = alert_counts.loc[common_indices]

    rolling_burden = alert_series.rolling(window='48h', closed='left').sum()
    
    df_features['recent_event_burden'] = rolling_burden.values
    
    df_features['recent_event_burden'] = df_features['recent_event_burden'].fillna(0)
    
    return df_context

## Section 10: Label Target Variables (Multi-Severity Alert Windows)

In [ ]:
def create_target_labels(df_features, senior_id, conn):
    """Create binary target labels for all severity levels based on alerts in the prior 24 hours."""
    
    if df_features is None or len(df_features) == 0:
        return df_features
    
    df_labeled = df_features.copy()
    df_labeled['label_1'] = 0
    df_labeled['label_2'] = 0
    df_labeled['label_3'] = 0
    
    for severity in [1, 2, 3]:
        query = """
        SELECT alert_date FROM alerts 
        WHERE senior_id = ? AND severity = ?
        """
        
        alerts_df = pd.read_sql_query(query, conn, params=(str(senior_id), severity))
        
        if len(alerts_df) == 0:
            continue
        
        alerts_df['alert_date'] = pd.to_datetime(alerts_df['alert_date'])
        alert_dates = alerts_df['alert_date'].values
        
        label_col = f'label_{severity}'
        for alert_ts in alert_dates:
            window_start = pd.Timestamp(alert_ts) - timedelta(hours=24)
            window_end = pd.Timestamp(alert_ts)
            
            mask = (df_labeled['timestamp'] >= window_start) & \
                   (df_labeled['timestamp'] < window_end)
            df_labeled.loc[mask, label_col] = 1
    
    return df_labeled

Target labeling function defined


## Section 11: Process Senior-by-Senior and Save to Parquet

In [12]:
# Gold Dataset seniors have already been loaded in Section 2
# They were saved from notebook 03 analysis

print("\n" + "=" * 100)
print("PROCESSING GOLD DATASET SENIORS")
print("=" * 100)
print(f"Total seniors to process: {len(gold_seniors_list):,}")
print(f"Sample seniors: {gold_seniors_list[:5]}")
print("=" * 100)


PROCESSING GOLD DATASET SENIORS
Total seniors to process: 8,268
Sample seniors: [np.int64(32776), np.int64(32788), np.int64(32821), np.int64(32832), np.int64(32834)]


In [13]:
## Quick diagnostics: inspect one senior's measurements before full run
sample_senior = gold_seniors_list[0] if len(gold_seniors_list) > 0 else None
if sample_senior is not None:
    print(f"Sampling senior: {sample_senior}")
    diag_query = """
    SELECT senior_id, type, value, sbp, dbp, date
    FROM measurements
    WHERE senior_id = ?
    ORDER BY date
    LIMIT 5
    """
    diag_df = pd.read_sql_query(diag_query, conn, params=(str(sample_senior),))
    print(f"Rows fetched: {len(diag_df)}")
    print(diag_df.head())
else:
    print("Gold seniors list is empty; cannot sample.")


Sampling senior: 32776
Rows fetched: 5
   senior_id   type  value   sbp   dbp                 date
0      32776  Steps   28.0  None  None  2025-11-01 08:11:36
1      32776  Steps   38.0  None  None  2025-11-01 08:17:36
2      32776  Steps   77.0  None  None  2025-11-01 08:20:36
3      32776  Steps  174.0  None  None  2025-11-01 08:23:36
4      32776  Steps  187.0  None  None  2025-11-01 08:26:36


In [14]:
# Coverage diagnostics: do we have measurements for any gold senior?
print("\n=== COVERAGE DIAGNOSTICS ===")
total_measurements = pd.read_sql_query("SELECT COUNT(*) AS c FROM measurements", conn)['c'][0]
distinct_measurement_seniors = pd.read_sql_query("SELECT COUNT(DISTINCT senior_id) AS c FROM measurements", conn)['c'][0]
print(f"Total rows in measurements: {total_measurements:,}")
print(f"Distinct seniors in measurements: {distinct_measurement_seniors:,}")
print(f"Gold seniors count: {len(gold_seniors_list):,}")

# Fetch measurement senior ids and compute overlap in Python (avoids huge SQL IN)
measurement_seniors = set(pd.read_sql_query("SELECT DISTINCT senior_id FROM measurements", conn)['senior_id'])
gold_seniors_set = set(gold_seniors_list)
overlap_seniors = list(measurement_seniors & gold_seniors_set)
print(f"Overlap seniors between measurements and gold list: {len(overlap_seniors):,}")

# If overlap exists, sample one and preview rows
if len(overlap_seniors) > 0:
    sample_overlap = overlap_seniors[0]
    print(f"Sampling overlapping senior: {sample_overlap}")
    preview_df = pd.read_sql_query(
        """
        SELECT senior_id, type, value, sbp, dbp, date
        FROM measurements
        WHERE senior_id = ?
        ORDER BY date
        LIMIT 5
        """,
        conn,
        params=(str(sample_overlap),)
    )
    print(f"Rows fetched: {len(preview_df)}")
    print(preview_df.head())
else:
    print("No overlap between gold seniors and measurements table. Confirm database path and gold list source.")



=== COVERAGE DIAGNOSTICS ===


Total rows in measurements: 70,428,508
Distinct seniors in measurements: 12,823
Gold seniors count: 8,268
Overlap seniors between measurements and gold list: 8,268
Sampling overlapping senior: 32776
Rows fetched: 5
   senior_id   type  value   sbp   dbp                 date
0      32776  Steps   28.0  None  None  2025-11-01 08:11:36
1      32776  Steps   38.0  None  None  2025-11-01 08:17:36
2      32776  Steps   77.0  None  None  2025-11-01 08:20:36
3      32776  Steps  174.0  None  None  2025-11-01 08:23:36
4      32776  Steps  187.0  None  None  2025-11-01 08:26:36


In [ ]:
def process_senior_complete_pipeline(senior_id, conn, verbose=False):
    """
    Complete pipeline for a single senior:
    1. Resample to 15-min buckets
    2. Pivot to wide format
    3. Apply forward fill imputation
    4. Engineer signal features
    5. Fuse clinical flags
    6. Add alert context
    7. Create target labels (label_1, label_2, label_3)
    """
    
    try:
        if verbose:
            print(f"    [Step 1/7] Resampling vitals...")
        # Step 1: Resample
        df_resampled = resample_senior_vitals(senior_id, conn, verbose=verbose)
        if df_resampled is None or len(df_resampled) == 0:
            if verbose:
                print(f"    [Step 1/7] No measurements found - skipping")
            return None
        if verbose:
            print(f"    [Step 1/7] ✓ Resampled to {len(df_resampled)} buckets")
        
        if verbose:
            print(f"    [Step 2/7] Pivoting to wide format...")
        # Step 2: Pivot (already in wide format from resampling)
        df_wide = pivot_to_wide_format(df_resampled)
        if df_wide is None:
            return None
        if verbose:
            print(f"    [Step 2/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 3/7] Forward fill imputation...")
        # Step 3: Forward fill imputation
        df_filled = apply_forward_fill_imputation(df_wide)
        if verbose:
            print(f"    [Step 3/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 4/7] Engineering signal features...")
        # Step 4: Engineer signal features
        df_signals = engineer_signal_features(df_filled)
        if verbose:
            print(f"    [Step 4/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 5/7] Loading clinical flags...")
        # Step 5: Fuse clinical flags
        clinical_flags = load_clinical_domain_flags(senior_id, conn)
        if clinical_flags is not None:
            df_signals = fuse_clinical_flags(df_signals, clinical_flags)
            if verbose:
                print(f"    [Step 5/7] ✓ Flags loaded and fused")
        else:
            if verbose:
                print(f"    [Step 5/7] ⚠ No clinical flags found")
        
        if verbose:
            print(f"    [Step 6/7] Adding alert context (48h burden)...")
        # Step 6: Add alert context
        df_context = add_alert_context_feature(df_signals, senior_id, conn)
        if verbose:
            print(f"    [Step 6/7] ✓ Complete")
        
        if verbose:
            print(f"    [Step 7/7] Creating target labels (label_1, label_2, label_3)...")
        # Step 7: Create target labels for all severity levels
        df_final = create_target_labels(df_context, senior_id, conn)
        if verbose:
            print(f"    [Step 7/7] ✓ Complete")
        
        return df_final
    
    except Exception as e:
        print(f"Error processing senior {senior_id}: {str(e)}")
        import traceback
        traceback.print_exc()
        return None

print("Complete pipeline function defined")

Complete pipeline function defined


In [ ]:
print("=" * 100)
print("BUILDING FINAL FEATURE MATRIX")
print("=" * 100)

# Process seniors in batches to manage memory
batch_size = 100
all_features_list = []
successful_seniors = 0
failed_seniors = 0

output_path = '../data/processed/multimodal_features.parquet'

start_time = time.time()

# Enable verbose mode for first 3 seniors to diagnose issues
for idx, senior_id in enumerate(gold_seniors_list):
    senior_start = time.time()
    verbose = (idx < 3)  # Verbose output for first 3 seniors only
    
    # More frequent progress updates, especially at the start
    if idx < 10 or (idx + 1) % 10 == 0:
        elapsed = time.time() - start_time
        print(f"Processing senior {idx + 1}/{len(gold_seniors_list)} (ID: {senior_id}) | Elapsed: {elapsed/60:.1f}min | Success: {successful_seniors} | Failed: {failed_seniors}")
    
    # Process this senior
    df_senior = process_senior_complete_pipeline(senior_id, conn, verbose=verbose)
    
    if df_senior is not None and len(df_senior) > 0:
        all_features_list.append(df_senior)
        successful_seniors += 1
        senior_elapsed = time.time() - senior_start
        if idx < 5:  # Show timing for first few seniors
            print(f"  ✓ Senior {senior_id} processed in {senior_elapsed:.2f}s - {len(df_senior)} time buckets")
    else:
        failed_seniors += 1
        if idx < 10:  # Show failures for first 10
            print(f"  ✗ Senior {senior_id} returned no data")

print(f"\n✓ Successfully processed: {successful_seniors:,} seniors")
print(f"✗ Failed/skipped: {failed_seniors:,} seniors")

# Combine all seniors into one large dataframe
if len(all_features_list) > 0:
    df_final_matrix = pd.concat(all_features_list, ignore_index=True)
    print(f"\nFinal Feature Matrix Shape: {df_final_matrix.shape}")
    print(f"Total Time Buckets: {len(df_final_matrix):,}")
    print(f"Unique Seniors: {df_final_matrix['senior_id'].nunique():,}")
    
    # Display column information
    print(f"\nFeature Columns ({len(df_final_matrix.columns)}):")
    for col in df_final_matrix.columns:
        print(f"  - {col}")
    
    # Display target distributions
    print(f"\nTarget Variable Distributions:")
    print(f"\n  label_1 (Severity 1):")
    print(df_final_matrix['label_1'].value_counts())
    print(f"\n  label_2 (Severity 2):")
    print(df_final_matrix['label_2'].value_counts())
    print(f"\n  label_3 (Severity 3):")
    print(df_final_matrix['label_3'].value_counts())
    
    print(f"\nData types:")
    print(df_final_matrix.dtypes)
else:
    print("No seniors were successfully processed!")

BUILDING FINAL FEATURE MATRIX
Processing senior 1/8268 (ID: 32776) | Elapsed: 0.0min | Success: 0 | Failed: 0
    [Step 1/7] Resampling vitals...
      → Found 1,025 measurement rows for senior 32776
    [Step 1/7] ✓ Resampled to 2838 buckets
    [Step 2/7] Pivoting to wide format...
    [Step 2/7] ✓ Complete
    [Step 3/7] Forward fill imputation...
    [Step 3/7] ✓ Complete
    [Step 4/7] Engineering signal features...
    [Step 4/7] ✓ Complete
    [Step 5/7] Loading clinical flags...
    [Step 5/7] ✓ Flags loaded and fused
    [Step 6/7] Adding alert context (48h burden)...
    [Step 6/7] ✓ Complete
    [Step 7/7] Creating target labels...
    [Step 7/7] ✓ Complete
  ✓ Senior 32776 processed in 0.66s - 2838 time buckets
Processing senior 2/8268 (ID: 32788) | Elapsed: 0.0min | Success: 1 | Failed: 0
    [Step 1/7] Resampling vitals...
      → Found 2,580 measurement rows for senior 32788
    [Step 1/7] ✓ Resampled to 2851 buckets
    [Step 2/7] Pivoting to wide format...
    [Step 2/

KeyboardInterrupt: 

In [23]:
print("=" * 100)
print("QUICK SANITY CHECK: PROCESS SUBSET OF SENIORS")
print("=" * 100)
print("This cell processes a small subset to verify the pipeline works before full run.")
print()

# ============================================================================
# CONFIGURATION: Adjust these values to control the subset size
# ============================================================================
subset_size = 100  # Process first N seniors (adjust as needed: 50, 100, 200, 500, etc.)
# ============================================================================

print(f"Processing first {subset_size:,} seniors from {len(gold_seniors_list):,} total...")
print()

# Process subset
subset_seniors = gold_seniors_list[:subset_size]
all_features_list_subset = []
successful_seniors_subset = 0
failed_seniors_subset = 0

subset_start_time = time.time()

for idx, senior_id in enumerate(subset_seniors):
    verbose = (idx < 3)  # Verbose for first 3
    
    # Progress update every 10 seniors
    if idx < 10 or (idx + 1) % 10 == 0:
        elapsed = time.time() - subset_start_time
        print(f"  [{idx + 1:,}/{subset_size:,}] Senior {senior_id} | Elapsed: {elapsed/60:.1f}min | Success: {successful_seniors_subset} | Failed: {failed_seniors_subset}")
    
    # Process this senior
    df_senior = process_senior_complete_pipeline(senior_id, conn, verbose=verbose)
    
    if df_senior is not None and len(df_senior) > 0:
        all_features_list_subset.append(df_senior)
        successful_seniors_subset += 1
    else:
        failed_seniors_subset += 1

print()
print("=" * 100)
print("INTERIM RESULTS (SUBSET)")
print("=" * 100)

if len(all_features_list_subset) > 0:
    df_interim = pd.concat(all_features_list_subset, ignore_index=True)
    
    print(f"\n✓ Successfully processed: {successful_seniors_subset:,}/{subset_size:,} seniors")
    print(f"✗ Failed/skipped: {failed_seniors_subset:,}/{subset_size:,} seniors")
    
    print(f"\n--- DATAFRAME SHAPE ---")
    print(f"Rows (time buckets): {len(df_interim):,}")
    print(f"Columns: {len(df_interim.columns)}")
    print(f"Total elements: {df_interim.size:,}")
    
    print(f"\n--- UNIQUE SENIORS ---")
    unique_seniors = df_interim['senior_id'].nunique()
    print(f"Unique seniors in interim: {unique_seniors:,}")
    print(f"Avg time buckets per senior: {len(df_interim) / max(1, unique_seniors):.0f}")
    
    print(f"\n--- TARGET LABEL DISTRIBUTIONS ---")
    for label_col in ['label_1', 'label_2', 'label_3']:
        print(f"\n  {label_col}:")
        label_dist = df_interim[label_col].value_counts().sort_index()
        for label, count in label_dist.items():
            pct = 100 * count / len(df_interim)
            print(f"    {label_col} = {label}: {count:,} ({pct:.2f}%)")
        
        # Show class imbalance ratio
        if len(label_dist) == 2:
            pos_count = label_dist.get(1, 0)
            neg_count = label_dist.get(0, 0)
            if pos_count > 0:
                imbalance_ratio = neg_count / pos_count
                print(f"    Imbalance ratio (0:1): {imbalance_ratio:.1f}:1")
    
    print(f"\n--- VITAL SIGNS COVERAGE ---")
    vital_cols = ['heartrate', 'temperature', 'saturation', 'steps', 'sbp', 'dbp']
    for col in vital_cols:
        if col in df_interim.columns:
            non_null = df_interim[col].notna().sum()
            pct_coverage = 100 * non_null / len(df_interim)
            print(f"  {col}: {non_null:,} / {len(df_interim):,} ({pct_coverage:.1f}%)")
    
    print(f"\n--- ENGINEERED FEATURES COVERAGE ---")
    engineered_cols = ['hr_volatility', 'bp_trend', 'pulse_pressure', 'recent_event_burden']
    for col in engineered_cols:
        if col in df_interim.columns:
            non_null = df_interim[col].notna().sum()
            pct_coverage = 100 * non_null / len(df_interim)
            print(f"  {col}: {non_null:,} / {len(df_interim):,} ({pct_coverage:.1f}%)")
    
    print(f"\n--- CLINICAL FLAGS COVERAGE ---")
    # Find clinical flag columns (not timestamp, senior_id, or known feature columns)
    known_cols = {'timestamp', 'senior_id', 'heartrate', 'temperature', 'saturation', 'steps', 
                  'sbp', 'dbp', 'hr_volatility', 'bp_trend', 'pulse_pressure', 'recent_event_burden', 'label_1', 'label_2', 'label_3'}
    flag_cols = [col for col in df_interim.columns if col not in known_cols]
    if len(flag_cols) > 0:
        for col in flag_cols[:5]:  # Show first 5 flag columns
            non_null = df_interim[col].notna().sum()
            pct_coverage = 100 * non_null / len(df_interim)
            print(f"  {col}: {non_null:,} / {len(df_interim):,} ({pct_coverage:.1f}%)")
        if len(flag_cols) > 5:
            print(f"  ... and {len(flag_cols) - 5} more flag columns")
    
    elapsed_subset = time.time() - subset_start_time
    print(f"\n--- PERFORMANCE ---")
    print(f"Total time: {elapsed_subset:.1f}s ({elapsed_subset/60:.2f}min)")
    print(f"Rate: {successful_seniors_subset / max(1, elapsed_subset):.1f} seniors/sec")
    
    print(f"\n--- MEMORY USAGE ---")
    memory_mb = df_interim.memory_usage(deep=True).sum() / (1024**2)
    print(f"Interim dataframe: {memory_mb:.1f} MB")
    
    print("\n" + "=" * 100)
    print("✓ SANITY CHECK COMPLETE")
    print("=" * 100)
    print(f"\nTo run the FULL pipeline on all {len(gold_seniors_list):,} seniors:")
    print(f"  1. Run the 'BUILDING FINAL FEATURE MATRIX' cell")
    print(f"  2. Then run 'SAVING TO PARQUET' cell")
    print()
else:
    print("✗ No seniors were successfully processed in subset!")
    print("Check the first few senior IDs and data availability.")

QUICK SANITY CHECK: PROCESS SUBSET OF SENIORS
This cell processes a small subset to verify the pipeline works before full run.

Processing first 100 seniors from 8,268 total...

  [1/100] Senior 32776 | Elapsed: 0.0min | Success: 0 | Failed: 0
    [Step 1/7] Resampling vitals...
      → Found 1,025 measurement rows for senior 32776
    [Step 1/7] ✓ Resampled to 2838 buckets
    [Step 2/7] Pivoting to wide format...
    [Step 2/7] ✓ Complete
    [Step 3/7] Forward fill imputation...
    [Step 3/7] ✓ Complete
    [Step 4/7] Engineering signal features...
    [Step 4/7] ✓ Complete
    [Step 5/7] Loading clinical flags...
    [Step 5/7] ✓ Flags loaded and fused
    [Step 6/7] Adding alert context (48h burden)...
    [Step 6/7] ✓ Complete
    [Step 7/7] Creating target labels...
    [Step 7/7] ✓ Complete
  [2/100] Senior 32788 | Elapsed: 0.0min | Success: 1 | Failed: 0
    [Step 1/7] Resampling vitals...
      → Found 2,580 measurement rows for senior 32788
    [Step 1/7] ✓ Resampled to 285

KeyError: 'label_1'

In [22]:
# Quick diagnostic: Check if gold seniors have ANY Severity 3 alerts
query = """
SELECT 
    COUNT(DISTINCT senior_id) as seniors_with_sev3,
    COUNT(*) as total_sev3_alerts
FROM alerts 
WHERE severity = 2 AND senior_id IN (
    SELECT DISTINCT senior_id FROM measurements
)
"""
sev3_stats = pd.read_sql_query(query, conn)
seniors_with_sev3 = sev3_stats['seniors_with_sev3'][0]
total_sev3_alerts = sev3_stats['total_sev3_alerts'][0]
print(f"\n✓ Severity 3 Alert Coverage Check:")
print(f"  Seniors with Severity 3 alerts: {seniors_with_sev3:,} out of {len(gold_seniors_list):,} gold seniors")
print(f"  Total Severity 3 alerts among gold seniors: {total_sev3_alerts:,}")


✓ Severity 3 Alert Coverage Check:
  Seniors with Severity 3 alerts: 213 out of 8,268 gold seniors
  Total Severity 3 alerts among gold seniors: 256


In [ ]:
# Save to Parquet
if len(all_features_list) > 0:
    print("\n" + "=" * 100)
    print("SAVING TO PARQUET")
    print("=" * 100)
    
    # Convert timestamp to string for better parquet compatibility
    df_final_matrix['timestamp'] = df_final_matrix['timestamp'].astype(str)
    
    # Save to parquet format (compressed)
    df_final_matrix.to_parquet(output_path, compression='snappy', index=False)
    
    print(f"✓ Feature matrix saved to: {output_path}")
    print(f"✓ File size: {np.round(pd.io.common.get_filepath_or_buffer(output_path)[0].__sizeof__() / (1024**3), 2)} GB")
    
    # Verify the file
    df_verify = pd.read_parquet(output_path)
    print(f"\n✓ Verification:")
    print(f"  - Rows: {len(df_verify):,}")
    print(f"  - Columns: {len(df_verify.columns)}")
    print(f"  - Memory usage: {df_verify.memory_usage(deep=True).sum() / (1024**3):.2f} GB")
    
    # Summary statistics
    print(f"\n" + "=" * 100)
    print("FINAL FEATURE MATRIX SUMMARY")
    print("=" * 100)
    print(f"Total Rows (Time Buckets): {len(df_verify):,}")
    print(f"Unique Seniors: {df_verify['senior_id'].nunique():,}")
    print(f"Date Range: {df_verify['timestamp'].min()} to {df_verify['timestamp'].max()}")
    print(f"\nTarget Variable Distributions:")
    for label_col in ['label_1', 'label_2', 'label_3']:
        print(f"\n  {label_col}:")
        label_counts = df_verify[label_col].value_counts()
        for label, count in label_counts.items():
            print(f"    {label_col} = {label}: {count:,} ({100*count/len(df_verify):.2f}%)")
else:
    print("Cannot save - no features were generated.")

## Section 12: Feature Matrix Exploration and Statistics

In [ ]:
if len(all_features_list) > 0:
    # Load the saved parquet file for exploration
    df_matrix = pd.read_parquet(output_path)
    
    print("=" * 100)
    print("FEATURE MATRIX DETAILED EXPLORATION")
    print("=" * 100)
    
    # Data types summary
    print("\nData Type Summary:")
    print(df_matrix.dtypes.value_counts())
    
    # Missing values
    print("\nMissing Values Summary:")
    missing = df_matrix.isnull().sum()
    if missing.sum() > 0:
        print(missing[missing > 0].sort_values(ascending=False))
    else:
        print("No missing values!")
    
    # Feature statistics
    print("\nVital Signs Statistics:")
    vital_cols = [col for col in df_matrix.columns 
                  if col in ['heartrate', 'temperature', 'saturation', 'steps', 'sbp', 'dbp']]
    if len(vital_cols) > 0:
        print(df_matrix[vital_cols].describe())
    
    # Engineered features statistics
    print("\nEngineered Features Statistics:")
    engineered_cols = [col for col in df_matrix.columns 
                       if col in ['hr_volatility', 'bp_trend', 'pulse_pressure', 'recent_event_burden']]
    if len(engineered_cols) > 0:
        print(df_matrix[engineered_cols].describe())
    
    # Target label statistics
    print("\nTarget Label Statistics:")
    for label_col in ['label_1', 'label_2', 'label_3']:
        pos_count = (df_matrix[label_col] == 1).sum()
        neg_count = (df_matrix[label_col] == 0).sum()
        pos_pct = 100 * pos_count / len(df_matrix)
        print(f"\n  {label_col}:")
        print(f"    Total samples: {len(df_matrix):,}")
        print(f"    Positive cases ({label_col}=1): {pos_count:,} ({pos_pct:.2f}%)")
        print(f"    Negative cases ({label_col}=0): {neg_count:,} ({100*neg_count/len(df_matrix):.2f}%)")
        if pos_count > 0:
            print(f"    Class balance ratio: 1:{neg_count / pos_count:.1f}")
    
    print("\n" + "=" * 100)
    print("FEATURE ENGINEERING COMPLETE")
    print("=" * 100)
    print(f"✓ Output file: {output_path}")
    print(f"✓ Total samples: {len(df_matrix):,}")
    print(f"✓ Total features: {len(df_matrix.columns)}")
    print(f"✓ Seniors processed: {df_matrix['senior_id'].nunique():,}")
    
    conn.close()
    print("\n✓ Database connection closed")